# Tweeting Fear — GPT-2 Fine-Tuning v2
### BUSN 20800 Big Data
**Hewitt Watkins · Erik Lopez · Mateo Fretes · Vedant Dangayach**

**Changes from v1:**
- Clean tweet text (URLs/media tokens stripped, ≥5 words)
- 6 LDA topics (vs 4)
- 5-bucket VIX/EPU conditioning (VERY_LOW → VERY_HIGH)
- Saves to `booth_results_v2/` in Drive

**Input:** `booth_final_data/tweet_labels_v2.csv`
**Output:** `booth_results_v2/` in Google Drive

> Run on a GPU runtime: Runtime → Change runtime type → T4 GPU

## 0. Mount Drive & Create Output Folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DATA_DIR    = '/content/drive/MyDrive/booth_final_data'
RESULTS_DIR = '/content/drive/MyDrive/booth_results_v2'
MODELS_DIR  = os.path.join(RESULTS_DIR, 'models', 'trump_gpt2_v2')
RESULTS_OUT = os.path.join(RESULTS_DIR, 'results')

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(RESULTS_OUT, exist_ok=True)

print('Drive mounted. Folders ready:')
print(f'  Data in:     {DATA_DIR}')
print(f'  Models out:  {MODELS_DIR}')
print(f'  Results out: {RESULTS_OUT}')

In [ ]:
import zipfile

zip_path    = os.path.join(DATA_DIR, 'data.zip')
labels_path = os.path.join(DATA_DIR, 'tweet_labels_v2.csv')

if os.path.exists(labels_path):
    print(f'tweet_labels_v2.csv already present — ready.')

elif os.path.exists(os.path.join(DATA_DIR, 'data', 'tweet_labels_v2.csv')):
    DATA_DIR    = os.path.join(DATA_DIR, 'data')
    labels_path = os.path.join(DATA_DIR, 'tweet_labels_v2.csv')
    print(f'Found in data/ subfolder — updated DATA_DIR to {DATA_DIR}')

else:
    if not os.path.exists(zip_path):
        raise FileNotFoundError(
            f'tweet_labels_v2.csv not found in {DATA_DIR}\n'
            'Please upload tweet_labels_v2.csv (output of unsupervised_v2.ipynb) to booth_final_data/ in Drive.'
        )
    print(f'Unzipping {zip_path} ...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DATA_DIR)
    if os.path.exists(os.path.join(DATA_DIR, 'data', 'tweet_labels_v2.csv')):
        DATA_DIR    = os.path.join(DATA_DIR, 'data')
        labels_path = os.path.join(DATA_DIR, 'tweet_labels_v2.csv')
    if not os.path.exists(labels_path):
        raise FileNotFoundError('tweet_labels_v2.csv not found after unzip.')

print(f'Using: {labels_path}')

## 1. Install & Import

In [ ]:
!pip install transformers datasets accelerate -q
print('Done')

In [ ]:
import pandas as pd
import numpy as np
import torch
import json
import random
import math
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    GPT2LMHeadModel, GPT2Tokenizer,
    Trainer, TrainingArguments,
    DataCollatorForLanguageModeling,
)
from datasets import Dataset as HFDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Change runtime to T4 GPU.')

## 2. Load Data & Inspect Label Distribution

In [ ]:
df = pd.read_csv(labels_path)
df = df.dropna(subset=['gpt2_prompt', 'vix_bucket', 'epu_bucket', 'lda_topic'])
df['lda_topic'] = df['lda_topic'].astype(int)
print(f'Loaded {len(df):,} labeled tweets')
print(f'Columns: {list(df.columns)}')
print()
print('VIX bucket counts:')
print(df['vix_bucket'].value_counts().sort_index().to_string())
print()
print('EPU bucket counts:')
print(df['epu_bucket'].value_counts().sort_index().to_string())
print()
print('Topic counts:')
print(df['lda_topic'].value_counts().sort_index().to_string())
print()
combo = df.groupby(['vix_bucket','epu_bucket','lda_topic']).size().reset_index(name='count')
print(f'Total conditioning combinations: {len(combo)}')
print(f'Sparsest combination: {combo["count"].min()} tweets')
print(f'Richest  combination: {combo["count"].max()} tweets')

In [ ]:
# Load cutoffs from v2 file, fall back to computing from data
cutoffs_path = os.path.join(DATA_DIR, 'bucket_cutoffs_v2.json')
if os.path.exists(cutoffs_path):
    with open(cutoffs_path) as f:
        cutoffs = json.load(f)
    print('Loaded bucket_cutoffs_v2.json')
else:
    # Recompute from weekly data in tweet_labels_v2.csv
    weekly = df.groupby('week_end')[['vix','epu']].first().dropna()
    vix_cuts = weekly['vix'].quantile([0.2, 0.4, 0.6, 0.8]).values
    epu_cuts = weekly['epu'].quantile([0.2, 0.4, 0.6, 0.8]).values
    cutoffs = {'vix': vix_cuts.tolist(), 'epu': epu_cuts.tolist(), 'n_bins': 5}
    print('Computed cutoffs from data (bucket_cutoffs_v2.json not found)')

with open(os.path.join(RESULTS_OUT, 'bucket_cutoffs_v2.json'), 'w') as f:
    json.dump(cutoffs, f, indent=2)

print(f'VIX cutoffs: {[round(x,2) for x in cutoffs["vix"]]}')
print(f'EPU cutoffs: {[round(x,2) for x in cutoffs["epu"]]}')
print('Saved bucket_cutoffs_v2.json to results/')

## 3. Tokenizer & Model Setup

5-bucket VIX/EPU special tokens + 6 topic tokens.

In [ ]:
MODEL_NAME = 'gpt2'
tokenizer  = GPT2Tokenizer.from_pretrained(MODEL_NAME)

topics = sorted(df['lda_topic'].unique())
print(f'Topics found: {topics}')

VIX_LABELS = ['VIX_VERY_LOW', 'VIX_LOW', 'VIX_MED', 'VIX_HIGH', 'VIX_VERY_HIGH']
EPU_LABELS = ['EPU_VERY_LOW', 'EPU_LOW', 'EPU_MED', 'EPU_HIGH', 'EPU_VERY_HIGH']

special_tokens = VIX_LABELS + EPU_LABELS + [f'[TOPIC_{i}]' for i in topics]
tokenizer.add_special_tokens({'additional_special_tokens': special_tokens})
tokenizer.pad_token = tokenizer.eos_token

print(f'Special tokens added: {special_tokens}')
print(f'Vocabulary size: {len(tokenizer):,}')

model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))
model.to(device)

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {params/1e6:.1f}M')
print('Model ready.')

## 4. Prepare Dataset

90/10 split. Format: `[VIX_X] [EPU_X] [TOPIC_N] <tweet><|endoftext|>`

In [ ]:
df['training_text'] = df['gpt2_prompt'].astype(str) + tokenizer.eos_token

random.seed(42)
indices   = list(range(len(df)))
random.shuffle(indices)
split_idx = int(0.9 * len(indices))
train_idx = indices[:split_idx]
test_idx  = indices[split_idx:]

train_texts = df.iloc[train_idx]['training_text'].tolist()
test_texts  = df.iloc[test_idx]['training_text'].tolist()
print(f'Train: {len(train_texts):,}  |  Test: {len(test_texts):,}')

def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)

train_ds = HFDataset.from_dict({'text': train_texts}).map(
    tokenize_fn, batched=True, remove_columns=['text'])
test_ds  = HFDataset.from_dict({'text': test_texts}).map(
    tokenize_fn, batched=True, remove_columns=['text'])

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print('Datasets ready.')

## 5. Fine-Tune

In [ ]:
training_args = TrainingArguments(
    output_dir=os.path.join(RESULTS_DIR, 'models', 'checkpoints_v2'),
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=200,
    weight_decay=0.01,
    learning_rate=5e-5,
    fp16=(device.type == 'cuda'),
    logging_dir=os.path.join(RESULTS_OUT, 'logs'),
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    save_total_limit=2,
    report_to='none',
    prediction_loss_only=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=collator,
)

print('Starting training...')
train_result = trainer.train()
print('Training complete.')
print(f'  Total steps: {train_result.global_step}')
print(f'  Train loss:  {train_result.training_loss:.4f}')

## 6. Save Model & Tokenizer to Drive

In [ ]:
model.save_pretrained(MODELS_DIR)
tokenizer.save_pretrained(MODELS_DIR)
print(f'Model saved to {MODELS_DIR}')

metrics = {
    'train_loss':    float(train_result.training_loss),
    'global_steps':  int(train_result.global_step),
    'train_samples': int(len(train_texts)),
    'test_samples':  int(len(test_texts)),
    'topics':        [int(t) for t in topics],
    'vix_labels':    VIX_LABELS,
    'epu_labels':    EPU_LABELS,
    'n_buckets':     5,
}
with open(os.path.join(RESULTS_OUT, 'training_metrics_v2.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved training_metrics_v2.json')

## 7. Evaluate — Perplexity

In [ ]:
eval_results = trainer.evaluate()
perplexity   = math.exp(eval_results['eval_loss'])
print(f'Eval loss:   {eval_results["eval_loss"]:.4f}')
print(f'Perplexity:  {perplexity:.2f}')

with open(os.path.join(RESULTS_OUT, 'perplexity_v2.txt'), 'w') as f:
    f.write(f'eval_loss:  {eval_results["eval_loss"]:.4f}\n')
    f.write(f'perplexity: {perplexity:.2f}\n')
print('Saved perplexity_v2.txt')

## 8. Generate Sample Tweets (B2 — All Topics per VIX×EPU)

All 25 VIX×EPU combinations × 6 topics = 150 sample tweets.

In [ ]:
model.eval()

def generate_tweet(vix_label, epu_label, topic_id,
                   max_new_tokens=80, temperature=0.85, top_p=0.92):
    prompt     = f'[{vix_label}] [{epu_label}] [TOPIC_{topic_id}]'
    input_ids  = tokenizer.encode(prompt, return_tensors='pt').to(device)
    prompt_len = input_ids.shape[1]
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][prompt_len:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

sample_rows = []
print('Generating sample tweets for all VIX x EPU x Topic combinations...')
for vl in VIX_LABELS:
    for el in EPU_LABELS:
        for t in topics:
            text = generate_tweet(vl, el, t)
            sample_rows.append({
                'vix_label': vl, 'epu_label': el,
                'topic': t, 'generated_tweet': text
            })
        print(f'  {vl} x {el}: done')

samples_df = pd.DataFrame(sample_rows)
samples_df.to_csv(os.path.join(RESULTS_OUT, 'sample_generations_v2.csv'), index=False)
print(f'Saved sample_generations_v2.csv — {len(samples_df)} generated tweets')

## 9. Preview Generated Tweets

In [ ]:
for vl in VIX_LABELS:
    for el in EPU_LABELS:
        subset = samples_df[(samples_df['vix_label']==vl) & (samples_df['epu_label']==el)]
        print(f'\n{"="*60}')
        print(f'  [{vl}] [{el}]')
        print(f'{"="*60}')
        for _, row in subset.iterrows():
            print(f'  [Topic {row["topic"]}] {row["generated_tweet"][:140]}')

## 10. Memorization Check

In [ ]:
import difflib

train_originals = df['gpt2_prompt'].astype(str).tolist()

def most_similar(generated_text):
    scores = [
        difflib.SequenceMatcher(None, generated_text.lower(), t.lower()).ratio()
        for t in train_originals
    ]
    best_idx = max(range(len(scores)), key=lambda i: scores[i])
    return scores[best_idx], train_originals[best_idx]

THRESHOLD = 0.85
flagged = 0
results = []
print(f'Memorization check (flag threshold: {THRESHOLD})\n')

for _, row in samples_df.iterrows():
    score, match = most_similar(row['generated_tweet'])
    flag = score > THRESHOLD
    if flag:
        flagged += 1
    results.append({'vix_label': row['vix_label'], 'epu_label': row['epu_label'],
                    'topic': row['topic'], 'similarity': round(score, 4),
                    'flagged': flag, 'nearest_training_tweet': match})
    marker = '  ⚠️  SUSPICIOUS' if flag else ''
    print(f'[{row["vix_label"]}][{row["epu_label"]}][Topic {row["topic"]}]  sim={score:.3f}{marker}')
    if flag:
        print(f'  Generated: {row["generated_tweet"][:120]}')
        print(f'  Nearest:   {match[:120]}\n')

print(f'\n{flagged}/{len(samples_df)} flagged  |  mean similarity: {sum(r["similarity"] for r in results)/len(results):.3f}')

pd.DataFrame(results).to_csv(os.path.join(RESULTS_OUT, 'memorization_check_v2.csv'), index=False)
print('Saved memorization_check_v2.csv')